In [2]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Prevents display windows from popping up and crashing
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.neural_network import MLPClassifier

print("🚀 Running lightweight memory-safe Neural Network engine...")

# Auto-detect file path layout
csv_filename = 'customer_churn_nn.csv'
if os.path.exists(csv_filename):
    data_path = csv_filename
elif os.path.exists(os.path.join('part_1_neural_network_analysis', csv_filename)):
    data_path = os.path.join('part_1_neural_network_analysis', csv_filename)
else:
    import glob
    found_paths = glob.glob(f'**/{csv_filename}', recursive=True)
    if found_paths:
        data_path = found_paths[0]
    else:
        print(f"❌ Error: Cannot locate '{csv_filename}' in this workspace.")
        sys.exit(1)

# Load data matrix
df = pd.read_csv(data_path)

# Isolate columns
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

# Encode categorical categories cleanly
categorical_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True).astype(float)

# Stratified split to protect class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, stratify=y, random_state=42
)

# Standardize values
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

os.makedirs('results', exist_ok=True)
records = []

# --- EXPERIMENT 1: Baseline (1 Hidden Layer, 16 nodes, ReLU) ---
mlp1 = MLPClassifier(hidden_layer_sizes=(16,), activation='relu', learning_rate_init=0.001, 
                     batch_size=32, max_iter=20, random_state=42, verbose=False)
mlp1.fit(X_train_scaled, y_train)
cm1 = confusion_matrix(y_test, mlp1.predict(X_test_scaled))
records.append({'Experiment': 'Exp 1: Baseline', 'Train Loss': round(mlp1.loss_, 4), 'Train Accuracy': round(mlp1.score(X_train_scaled, y_train), 4), 'Test Loss': 'N/A', 'Test Accuracy': round(mlp1.score(X_test_scaled, y_test), 4)})

# --- EXPERIMENT 2: Deeper/Wider (2 Hidden Layers: 32 -> 16, High LR) ---
mlp2 = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', learning_rate_init=0.005, 
                     batch_size=64, max_iter=30, random_state=42, verbose=False)
mlp2.fit(X_train_scaled, y_train)
cm2 = confusion_matrix(y_test, mlp2.predict(X_test_scaled))
records.append({'Experiment': 'Exp 2: Deeper', 'Train Loss': round(mlp2.loss_, 4), 'Train Accuracy': round(mlp2.score(X_train_scaled, y_train), 4), 'Test Loss': 'N/A', 'Test Accuracy': round(mlp2.score(X_test_scaled, y_test), 4)})

# --- EXPERIMENT 3: Alternative (1 Hidden Layer, 16 nodes, Tanh) ---
mlp3 = MLPClassifier(hidden_layer_sizes=(16,), activation='tanh', learning_rate_init=0.0005, 
                     batch_size=16, max_iter=15, random_state=42, verbose=False)
mlp3.fit(X_train_scaled, y_train)
cm3 = confusion_matrix(y_test, mlp3.predict(X_test_scaled))
records.append({'Experiment': 'Exp 3: Alternative', 'Train Loss': round(mlp3.loss_, 4), 'Train Accuracy': round(mlp3.score(X_train_scaled, y_train), 4), 'Test Loss': 'N/A', 'Test Accuracy': round(mlp3.score(X_test_scaled, y_test), 4)})

# Export the hyperparameter comparison table
pd.DataFrame(records).to_csv('results/model_comparison_table.csv', index=False)

# Render and save evaluation graphs silently without crashing
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot training loss tracking curves
axes[0, 0].plot(mlp1.loss_curve_, label='Exp 1 Loss Curve', color='blue')
axes[0, 0].plot(mlp2.loss_curve_, label='Exp 2 Loss Curve', color='orange')
axes[0, 0].set_title('Training Loss Convergence Trajectory')
axes[0, 0].set_xlabel('Iterations')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Confusion Matrices
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1], cbar=False)
axes[0, 1].set_title('Exp 1 Confusion Matrix')
sns.heatmap(cm2, annot=True, fmt='d', cmap='Oranges', ax=axes[1, 0], cbar=False)
axes[1, 0].set_title('Exp 2 Confusion Matrix')
sns.heatmap(cm3, annot=True, fmt='d', cmap='Greens', ax=axes[1, 1], cbar=False)
axes[1, 1].set_title('Exp 3 Confusion Matrix')

plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=300)
plt.close(fig)

print("\n✅ Run Finished Successfully with Zero Memory Leaks!")
print("-> Saved metrics to: results/model_comparison_table.csv")
print("-> Saved visualizations to: results/evaluation_outputs.png")
print("\nYou are completely clear to submit your repository link now.")

ModuleNotFoundError: No module named 'pandas'